# CITCA 2026 · Optimización de hiperparámetros en redes neuronales
**Taller práctico (60 min) · Google Colab · TensorFlow/Keras + Optuna**

**Objetivo:** comparar ajuste manual, búsqueda en rejilla, aleatoria, optimización secuencial TPE y búsqueda simplificada de arquitecturas (NAS), usando Fashion-MNIST.

**Antes de comenzar:** Entorno de ejecución → Ejecutar todas las celdas por secciones. CPU es suficiente; los tiempos dependen del entorno. Las cifras se calculan, no están prefijadas.

**Cronograma:** 0–5 introducción; 5–12 modelo base; 12–20 manual; 20–30 rejilla/aleatoria; 30–43 Optuna; 43–52 NAS; 52–57 comparación; 57–60 cierre. La cuantización es opcional.

**Metodología:** los hiperparámetros se seleccionan exclusivamente con validación; el test se utiliza al final. Se utiliza un subconjunto de entrenamiento para que el ejercicio sea breve. Los resultados no constituyen una evaluación estadística exhaustiva.

## 1. Preparación (min 0–5)
Los **parámetros** (pesos y sesgos) se aprenden mediante entrenamiento; los **hiperparámetros** (tasa de aprendizaje, neuronas, capas, dropout, batch size) se eligen antes o durante la búsqueda.

In [ ]:
%pip -q install optuna
import os, time, random, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
SEED=42
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:',tf.__version__,'Optuna:',optuna.__version__)

## 2. Datos Fashion-MNIST (min 5–8)
Se usan 8,000 ejemplos para entrenamiento y 2,000 para validación, seleccionados de las 60,000 imágenes de entrenamiento. Las 10,000 imágenes de prueba quedan intactas.

In [ ]:
(x_all,y_all),(x_test,y_test)=tf.keras.datasets.fashion_mnist.load_data()
labels=['Camiseta','Pantalón','Suéter','Vestido','Abrigo','Sandalia','Camisa','Tenis','Bolso','Botín']
x_all=x_all.astype('float32').reshape(-1,784)/255.
x_test=x_test.astype('float32').reshape(-1,784)/255.
idx_train,idx_val=train_test_split(np.arange(len(y_all)),train_size=8000,test_size=2000,stratify=y_all,random_state=SEED)
x_train,y_train=x_all[idx_train],y_all[idx_train]
x_val,y_val=x_all[idx_val],y_all[idx_val]
fig,axes=plt.subplots(2,5,figsize=(10,4))
for i,ax in enumerate(axes.flat):
 ax.imshow(x_train[i].reshape(28,28),cmap='gray');ax.set_title(labels[y_train[i]]);ax.axis('off')
plt.tight_layout();plt.show()
print('Entrenamiento:',x_train.shape,'Validación:',x_val.shape,'Prueba:',x_test.shape)

## 3. Función común de entrenamiento
Para comparar métodos, se utiliza la misma partición, máximo de 4 épocas, batch size fijo y una función de evaluación. El tiempo registrado incluye construcción y entrenamiento. El early stopping se reserva para el reentrenamiento final.

In [ ]:
EPOCHS=4
registros=[]
def construir_modelo(capas=(64,32),lr=0.001,dropout=0.0):
 tf.keras.backend.clear_session()
 model=tf.keras.Sequential([tf.keras.Input(shape=(784,))])
 for unidades in capas:
  model.add(tf.keras.layers.Dense(int(unidades),activation='relu'))
  if dropout>0: model.add(tf.keras.layers.Dropout(float(dropout)))
 model.add(tf.keras.layers.Dense(10,activation='softmax'))
 model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=float(lr)),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
 return model

def evaluar(metodo,capas=(64,32),lr=0.001,dropout=0.,epochs=EPOCHS):
 tf.keras.utils.set_random_seed(SEED)
 inicio=time.perf_counter()
 model=construir_modelo(capas,lr,dropout)
 history=model.fit(x_train,y_train,validation_data=(x_val,y_val),epochs=epochs,batch_size=128,verbose=0)
 duracion=time.perf_counter()-inicio
 resultado=dict(metodo=metodo,capas=str(tuple(capas)),lr=lr,dropout=dropout,val_accuracy=max(history.history['val_accuracy']),segundos=duracion,parametros=model.count_params())
 registros.append(resultado)
 return resultado,history,model

## 4. Línea base (min 8–12)
**Pregunta:** ¿qué exactitud obtiene una MLP sin optimización?

In [ ]:
base,hist,modelo_base=evaluar('Base')
print(pd.Series(base))
plt.plot(hist.history['accuracy'],label='Entrenamiento');plt.plot(hist.history['val_accuracy'],label='Validación')
plt.xlabel('Época');plt.ylabel('Exactitud');plt.legend();plt.grid(alpha=.2);plt.show()

## 5. Búsqueda manual (min 12–20)
Cambia `CAPAS`, `LR` o `DROPOUT` y ejecuta nuevamente. El resultado se agrega a la tabla. Prueba, por ejemplo, (128,64), 0.001, 0.2.

In [ ]:
CAPAS=(128,64)
LR=0.001
DROPOUT=0.2
manual,_,_=evaluar('Manual',CAPAS,LR,DROPOUT)
print(pd.Series(manual))

## 6. Grid Search (min 20–25)
Evaluamos **cuatro combinaciones** de dos tamaños de capa y dos tasas de aprendizaje. El número de evaluaciones crece como el producto de las opciones.

In [ ]:
for capas,lr in itertools.product([(32,),(64,)], [0.001,0.003]):
 resultado,_,_=evaluar('Grid',capas,lr)
 print(capas,lr,round(resultado['val_accuracy'],4))

## 7. Random Search (min 25–30)
Se evalúan cuatro configuraciones extraídas sin reemplazo de un espacio de búsqueda mayor. El presupuesto es igual al de Grid Search, aunque los espacios no son idénticos.

In [ ]:
espacio=list(itertools.product([(32,),(64,),(128,64)],[0.0005,0.001,0.003],[0.,0.2]))
for capas,lr,dropout in random.Random(SEED).sample(espacio,4):
 resultado,_,_=evaluar('Random',capas,lr,dropout)
 print(capas,lr,dropout,round(resultado['val_accuracy'],4))

## 8. Optuna: optimización secuencial con TPE (min 30–43)
Optuna usa **TPESampler**, que estima distribuciones de configuraciones prometedoras. Con solo 6 ensayos, la comparación es demostrativa; TPE puede permanecer en su fase inicial de muestreo aleatorio. Para observar su comportamiento adaptativo, prueba `n_trials=15` después del taller.

In [ ]:
def objetivo(trial):
 capas=(trial.suggest_categorical('neuronas', [32,64,128]),)
 lr=trial.suggest_float('lr',0.0003,0.005,log=True)
 dropout=trial.suggest_categorical('dropout',[0.,0.2,0.4])
 resultado,_,_=evaluar('Optuna TPE',capas,lr,dropout)
 return resultado['val_accuracy']
study=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=SEED,n_startup_trials=3))
study.optimize(objetivo,n_trials=6,show_progress_bar=False)
print('Mejor validación:',study.best_value,'Mejores hiperparámetros:',study.best_params)
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.tight_layout();plt.show()

## 9. NAS simplificado (min 43–52)
La búsqueda de arquitectura también es optimización de hiperparámetros estructurales. Aquí se varían **profundidad** (1–3 capas) y **anchura** (32/64/128 neuronas). No es NAS avanzado (no hay búsqueda diferenciable ni evolución).

In [ ]:
def objetivo_nas(trial):
 profundidad=trial.suggest_int('capas',1,3)
 ancho=trial.suggest_categorical('ancho',[32,64,128])
 capas=tuple([ancho]*profundidad)
 resultado,_,_=evaluar('NAS',capas,0.001,0.1)
 trial.set_user_attr('parametros',resultado['parametros'])
 return resultado['val_accuracy']
study_nas=optuna.create_study(direction='maximize',sampler=optuna.samplers.RandomSampler(seed=SEED))
study_nas.optimize(objetivo_nas,n_trials=5,show_progress_bar=False)
print('Mejor arquitectura:',study_nas.best_params,'Accuracy validación:',study_nas.best_value)
print('Parámetros:',study_nas.best_trial.user_attrs['parametros'])

## 10. Comparación y evaluación final (min 52–57)
**Advertencia:** un método con más ensayos tiene más oportunidades de encontrar una buena configuración. Compara también tiempo total y número de entrenamientos; evita atribuir diferencias pequeñas a superioridad estadística.

In [ ]:
df=pd.DataFrame(registros)
resumen=df.groupby('metodo').agg(ensayos=('val_accuracy','size'),mejor_val_accuracy=('val_accuracy','max'),tiempo_total_s=('segundos','sum')).sort_values('mejor_val_accuracy',ascending=False)
display(resumen)
resumen['mejor_val_accuracy'].plot.bar(figsize=(9,4),ylim=(0.5,1),title='Mejor exactitud de validación por estrategia')
plt.ylabel('Exactitud');plt.tight_layout();plt.show()
resumen['tiempo_total_s'].plot.bar(figsize=(9,3),title='Tiempo total por estrategia')
plt.ylabel('Segundos');plt.tight_layout();plt.show()

### Evaluación final en test (una sola vez)
Seleccionamos la configuración de mayor exactitud de validación, reentrenamos con las mismas 8,000 imágenes y evaluamos el conjunto de prueba. Esta evaluación no debe utilizarse para volver a elegir hiperparámetros.

In [ ]:
mejor=df.loc[df['val_accuracy'].idxmax()]
import ast
capas=ast.literal_eval(mejor['capas'])
tf.keras.utils.set_random_seed(SEED)
modelo_final=construir_modelo(capas,mejor['lr'],mejor['dropout'])
modelo_final.fit(x_train,y_train,validation_data=(x_val,y_val),epochs=10,batch_size=128,verbose=0,callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=2,restore_best_weights=True)])
loss,acc=modelo_final.evaluate(x_test,y_test,verbose=0)
print('Configuración:',mejor[['metodo','capas','lr','dropout']].to_dict())
print(f'Exactitud en test: {acc:.4f}; parámetros: {modelo_final.count_params():,}')
y_pred=np.argmax(modelo_final.predict(x_test,batch_size=512,verbose=0),axis=1)
fig,ax=plt.subplots(figsize=(8,7));ConfusionMatrixDisplay(confusion_matrix(y_test,y_pred),display_labels=labels).plot(ax=ax,xticks_rotation=45,colorbar=False);plt.tight_layout();plt.show()

## 11. Extensión opcional: cuantización TFLite (fuera de los 60 min)
Convertimos el modelo FP32 y una versión con cuantización dinámica. Esta segunda versión no garantiza INT8 completo ni aceleración: para cuantización entera completa se requiere calibración representativa y configurar tipos de entrada/salida. Compararemos únicamente tamaño, no latencia.

In [ ]:
import pathlib
converter=tf.lite.TFLiteConverter.from_keras_model(modelo_final)
fp32=converter.convert()
converter=tf.lite.TFLiteConverter.from_keras_model(modelo_final)
converter.optimizations=[tf.lite.Optimize.DEFAULT]
optimizado=converter.convert()
pathlib.Path('fashion_mlp_fp32.tflite').write_bytes(fp32)
pathlib.Path('fashion_mlp_optimizado.tflite').write_bytes(optimizado)
print(pd.DataFrame({'modelo':['FP32','Optimizado TFLite'],'KiB':[len(fp32)/1024,len(optimizado)/1024]}))

## 12. Discusión (min 57–60)
1. ¿Cuál método encontró la mejor exactitud de validación y cuál consumió menos tiempo total?
2. ¿Por qué más neuronas no garantizan mejores resultados?
3. ¿Qué cambia al introducir dropout?
4. ¿Por qué una sola ejecución no demuestra que un método sea superior?
5. ¿Por qué la cuantización se considera una optimización de despliegue?

**Para ampliar después del taller:** repetir con 3–5 semillas; aumentar ensayos de TPE; usar poda (`MedianPruner`); comparar coste de inferencia; búsqueda multiobjetivo exactitud–latencia.

**Referencias:** [TensorFlow Fashion-MNIST](https://www.tensorflow.org/tutorials/keras/classification) · [Optuna](https://optuna.readthedocs.io/) · [TensorFlow Lite](https://www.tensorflow.org/lite/performance/post_training_quantization).